In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import sys
sys.path.append("..")
from config import *
from metric import *

In [2]:
def compute_and_print_statistics(df, is_embedding=False):
    rows = []
    
    for algorithm in ALGORITHMS:
        row_data = {'Algorithm': algorithm}
        
        ro_statistics = df.groupby(['type'])[algorithm].agg(['mean', 'std'])
        
        for type_label in ro_statistics.index:
            mean_val = ro_statistics.loc[type_label, 'mean']
            std_val = ro_statistics.loc[type_label, 'std']
            row_data[type_label] = f"{mean_val:.3f} ({std_val:.3f})"
        rows.append(row_data)
    
    results_df = pd.DataFrame(rows)
    if is_embedding:
        # reorder columns since 'text-embedding-3-large' comes first when ordered lexicographically
        results_df = results_df[['Algorithm', 'nomic-embed-text', 'text-embedding-3-small', 'text-embedding-3-large']]
    
    num_cols = len(results_df.columns)
    col_fmt = 'l' + 'r' * (num_cols - 1)
    
    latex_output = results_df.to_latex(index=False, escape=False, column_format=col_fmt)
    print(latex_output)

In [ ]:
def calculate_friedman_p(df, alg):
    grouped = df.groupby(['benchmark', 'strong', 'weak', 'type'])[alg].mean()
    pivoted_data = grouped.unstack(level='type')
    
    valid_pairs = pivoted_data.dropna()
    
    if valid_pairs.empty or valid_pairs.shape[1] < 2:
        return 1.0

    try:
        _, p_val = stats.friedmanchisquare(*[valid_pairs[col] for col in valid_pairs.columns])
        return p_val
    except ValueError:
        return 1.0

def compute_and_print_statistics_with_friedman(df, is_embedding=False):
    rows = []
    
    for algorithm in ALGORITHMS:
        row_data = {'Algorithm': algorithm}
        
        ro_statistics = df.groupby(['type'])[algorithm].agg(['mean', 'std'])
        
        for type_label in ro_statistics.index:
            mean_val = ro_statistics.loc[type_label, 'mean']
            std_val = ro_statistics.loc[type_label, 'std']
            row_data[type_label] = f"{mean_val:.3f} ({std_val:.3f})"
        
        p_val = calculate_friedman_p(df, algorithm)
        row_data['p-val'] = f"{p_val:.3f}" if p_val >= 0.001 else "$<$ 0.001"
        rows.append(row_data)
    
    results_df = pd.DataFrame(rows)
    if is_embedding:
        # reorder columns since 'text-embedding-3-large' comes first when ordered lexicographically
        results_df = results_df[['Algorithm', 'nomic-embed-text', 'text-embedding-3-small', 'text-embedding-3-large', 'p-val']]
    
    num_cols = len(results_df.columns)
    col_fmt = 'l' + 'r' * (num_cols - 1)
    
    latex_output = results_df.to_latex(index=False, escape=False, column_format=col_fmt)
    print(latex_output)

## RQ3-1 Does the selection of the specific Transformer layer for state extraction influence router performance?

In [4]:
testing_df = pd.read_csv(f'metrics/router_optimality_{DEFAULT_PROMPT_TYPE}.csv', index_col=0)
testing_df = testing_df[testing_df['input'] == 'internal state']
compute_and_print_statistics(testing_df)

\begin{tabular}{lrrr}
\toprule
Algorithm & 1/2 & 2/3 & 3/4 \\
\midrule
Logistic Regression & 0.536 (0.052) & 0.536 (0.051) & 0.539 (0.055) \\
SVM & 0.512 (0.049) & 0.511 (0.050) & 0.514 (0.055) \\
MLP & 0.529 (0.043) & 0.535 (0.044) & 0.537 (0.046) \\
Ensemble & 0.530 (0.049) & 0.536 (0.049) & 0.535 (0.056) \\
RouteLLM & 0.532 (0.050) & 0.532 (0.051) & 0.533 (0.051) \\
\bottomrule
\end{tabular}



In [5]:
compute_and_print_statistics_with_friedman(testing_df)

\begin{tabular}{lrrrr}
\toprule
Algorithm & 1/2 & 2/3 & 3/4 & p-val \\
\midrule
Logistic Regression & 0.536 (0.052) & 0.536 (0.051) & 0.539 (0.055) & 0.079 \\
SVM & 0.512 (0.049) & 0.511 (0.050) & 0.514 (0.055) & 0.571 \\
MLP & 0.529 (0.043) & 0.535 (0.044) & 0.537 (0.046) & 0.219 \\
Ensemble & 0.530 (0.049) & 0.536 (0.049) & 0.535 (0.056) & 0.295 \\
RouteLLM & 0.532 (0.050) & 0.532 (0.051) & 0.533 (0.051) & 0.869 \\
\bottomrule
\end{tabular}



In [6]:
code_gen_df = pd.read_csv(f'metrics/router_optimality_{DEFAULT_PROMPT_TYPE}_codegen.csv', index_col=0)
code_gen_df = code_gen_df[code_gen_df['input'] == 'internal state']
compute_and_print_statistics_with_friedman(code_gen_df)

\begin{tabular}{lrrrr}
\toprule
Algorithm & 1/2 & 2/3 & 3/4 & p-val \\
\midrule
Logistic Regression & 0.648 (0.148) & 0.670 (0.146) & 0.674 (0.134) & 0.004 \\
SVM & 0.649 (0.143) & 0.667 (0.145) & 0.657 (0.150) & 0.549 \\
MLP & 0.627 (0.114) & 0.644 (0.113) & 0.652 (0.093) & 0.043 \\
Ensemble & 0.654 (0.138) & 0.673 (0.141) & 0.683 (0.127) & 0.035 \\
RouteLLM & 0.639 (0.124) & 0.645 (0.106) & 0.654 (0.100) & 0.043 \\
\bottomrule
\end{tabular}



## RQ3-2. Does the choice of embedding model influence router performance?

In [7]:
testing_df = pd.read_csv(f'metrics/router_optimality_{DEFAULT_PROMPT_TYPE}.csv', index_col=0)
testing_df = testing_df[testing_df['input'] == 'prompt embedding']
compute_and_print_statistics(testing_df, is_embedding=True)

\begin{tabular}{lrrr}
\toprule
Algorithm & nomic-embed-text & text-embedding-3-small & text-embedding-3-large \\
\midrule
Logistic Regression & 0.484 (0.043) & 0.522 (0.042) & 0.522 (0.042) \\
SVM & 0.481 (0.040) & 0.502 (0.040) & 0.501 (0.040) \\
MLP & 0.501 (0.038) & 0.529 (0.053) & 0.531 (0.050) \\
Ensemble & 0.491 (0.037) & 0.525 (0.046) & 0.527 (0.045) \\
RouteLLM & 0.502 (0.029) & 0.524 (0.039) & 0.524 (0.039) \\
\bottomrule
\end{tabular}



In [8]:
compute_and_print_statistics_with_friedman(testing_df, is_embedding=True)

\begin{tabular}{lrrrr}
\toprule
Algorithm & nomic-embed-text & text-embedding-3-small & text-embedding-3-large & p-val \\
\midrule
Logistic Regression & 0.484 (0.043) & 0.522 (0.042) & 0.522 (0.042) & $<$ 0.001 \\
SVM & 0.481 (0.040) & 0.502 (0.040) & 0.501 (0.040) & 0.002 \\
MLP & 0.501 (0.038) & 0.529 (0.053) & 0.531 (0.050) & $<$ 0.001 \\
Ensemble & 0.491 (0.037) & 0.525 (0.046) & 0.527 (0.045) & $<$ 0.001 \\
RouteLLM & 0.502 (0.029) & 0.524 (0.039) & 0.524 (0.039) & $<$ 0.001 \\
\bottomrule
\end{tabular}



In [9]:
code_gen_df = pd.read_csv(f'metrics/router_optimality_{DEFAULT_PROMPT_TYPE}_codegen.csv', index_col=0)
code_gen_df = code_gen_df[code_gen_df['input'] == 'prompt embedding']
compute_and_print_statistics_with_friedman(code_gen_df, is_embedding=True)

\begin{tabular}{lrrrr}
\toprule
Algorithm & nomic-embed-text & text-embedding-3-small & text-embedding-3-large & p-val \\
\midrule
Logistic Regression & 0.576 (0.097) & 0.608 (0.107) & 0.608 (0.107) & 0.022 \\
SVM & 0.596 (0.110) & 0.608 (0.101) & 0.609 (0.101) & 0.819 \\
MLP & 0.604 (0.092) & 0.616 (0.069) & 0.623 (0.073) & 0.259 \\
Ensemble & 0.598 (0.097) & 0.623 (0.099) & 0.631 (0.100) & 0.165 \\
RouteLLM & 0.599 (0.068) & 0.625 (0.069) & 0.625 (0.069) & 0.004 \\
\bottomrule
\end{tabular}



## RQ3-1 & 3-2 Boxplot

In [10]:
plt.rcParams.update({'font.size': 18})

testing_df = pd.read_csv(f'metrics/router_optimality_{DEFAULT_PROMPT_TYPE}.csv', index_col=0)

plot_order = ['1/2', '2/3', '3/4', 'GAP', 
              'nomic-embed-text', 'text-embedding-3-small', 'text-embedding-3-large']

c1 = plt.get_cmap('Set2').colors[0]
c2 = plt.get_cmap('Set2').colors[1]
palette = {
    '1/2': c1, '2/3': c1, '3/4': c1, 
    'GAP': 'white', # Dummy color
    'nomic-embed-text': c2, 
    'text-embedding-3-small': c2, 
    'text-embedding-3-large': c2
}

for alg in ALGORITHMS:
    fig, ax = plt.subplots(figsize=(12, 4))
    
    sns.boxplot(data=testing_df, x='type', y=alg, ax=ax,
                order=plot_order, palette=palette, hue='type', legend=False)


    ax.grid(True, axis='y', linestyle='--', alpha=0.6)
    ax.set_axisbelow(True)
    
    ax.set_xlabel('')
    ax.set_xticks([])
    
    ax.axvline(x=3, color='gray', linestyle='--', linewidth=2, alpha=0.5)

    labels_map = {
        0: '1/2', 1: '2/3', 2: '3/4',
        4: 'nomic-embed-text', 
        5: 'text-embedding-3-small', 
        6: 'text-embedding-3-large'
    }
    
    y_lims = ax.get_ylim()
    y_range = y_lims[1] - y_lims[0]
    default_y = y_lims[0] - y_range * 0.05
    staggered_y = y_lims[0] - y_range * 0.15
    
    for idx, label in labels_map.items():
        y_pos = staggered_y if label == 'text-embedding-3-small' else default_y
        ax.text(idx, y_pos, label, ha='center', va='top', fontsize=16)

    ax.set_ylabel('Router Optimality', fontsize=18)
    sns.despine(ax=ax, top=True, right=True, left=True)
    
    ax.tick_params(axis='y', left=False)

    plt.tight_layout()
    plt.savefig(f'figures/RQ3_input_comparison_{alg}.pdf')
    plt.close()

## RQ3-3. Does the formatting of the input prompt impact the routing signal?

In [11]:
def filter_by_type(df, input_type):
    return df[df['type'] == input_type]

def calculate_wilcoxon_p(df, alg):
    grouped = df.groupby(['benchmark', 'strong', 'weak', 'prompt_style'])[alg].mean()
    paired_data = grouped.unstack(level='prompt_style')

    if 'entire' not in paired_data.columns or 'input only' not in paired_data.columns:
        return 1.0

    valid_pairs = paired_data.dropna(subset=['entire', 'input only'])
    
    if valid_pairs.empty:
        return 1.0

    entire = valid_pairs['entire']
    input_only = valid_pairs['input only']
    
    try:
        # Null Hypothesis: RO for input only is equal to RO for entire
        _, p_val = stats.wilcoxon(input_only, entire, alternative='two-sided', zero_method='zsplit')
        return p_val
    except ValueError:
        return 1.0

def compute_and_print_statistics_with_wilcoxon(testing_df, code_gen_df):
    rows = []
    
    for algorithm in ALGORITHMS:
        row_data = {'Algorithm': algorithm}
        
        ro_statistics = testing_df.groupby(['prompt_style'])[algorithm].agg(['mean', 'std'])
        entire_stats = ro_statistics.loc['entire']
        input_only_stats = ro_statistics.loc['input only']
        
        p_val = calculate_wilcoxon_p(testing_df, algorithm)
        
        row_data['Testing Entire'] = f"{entire_stats['mean']:.3f} ({entire_stats['std']:.3f})"
        row_data['Testing Input Only'] = f"{input_only_stats['mean']:.3f} ({input_only_stats['std']:.3f})"
        row_data['Testing p-val'] = f"{p_val:.3f}" if p_val >= 0.001 else "$<$ 0.001"
        
        ro_statistics = code_gen_df.groupby(['prompt_style'])[algorithm].agg(['mean', 'std'])
        entire_stats = ro_statistics.loc['entire']
        input_only_stats = ro_statistics.loc['input only']
        
        p_val = calculate_wilcoxon_p(code_gen_df, algorithm)
        
        row_data['Code Generation Entire'] = f"{entire_stats['mean']:.3f} ({entire_stats['std']:.3f})"
        row_data['Code Generation Input Only'] = f"{input_only_stats['mean']:.3f} ({input_only_stats['std']:.3f})"
        row_data['Code Generation p-val'] = f"{p_val:.3f}" if p_val >= 0.001 else "$<$ 0.001"
        
        rows.append(row_data)
    
    results_df = pd.DataFrame(rows)
    
    latex_output = results_df.to_latex(index=False, escape=False, column_format='lrrrrrr')
    print(latex_output)

In [12]:
def load_dataframes_for_prompt_style(is_code_gen, type_filter):
    if not is_code_gen:
        entire_df = pd.read_csv('metrics/router_optimality_entire.csv', index_col=0)
        input_only_df = pd.read_csv('metrics/router_optimality_input_only.csv', index_col=0)
    else:
        entire_df = pd.read_csv('metrics/router_optimality_entire_codegen.csv', index_col=0)
        input_only_df = pd.read_csv('metrics/router_optimality_input_only_codegen.csv', index_col=0)

    entire_df = filter_by_type(entire_df, type_filter)
    entire_df['prompt_style'] = 'entire'
    input_only_df = filter_by_type(input_only_df, type_filter)
    input_only_df['prompt_style'] = 'input only'

    return pd.concat([entire_df, input_only_df])

#### Internal State

In [13]:
testing_df = load_dataframes_for_prompt_style(is_code_gen=False, type_filter=DEFAULT_INTERNAL_STATE_TYPE)
code_gen_df = load_dataframes_for_prompt_style(is_code_gen=True, type_filter=DEFAULT_INTERNAL_STATE_TYPE)
compute_and_print_statistics_with_wilcoxon(testing_df, code_gen_df)

\begin{tabular}{lrrrrrr}
\toprule
Algorithm & Testing Entire & Testing Input Only & Testing p-val & Code Generation Entire & Code Generation Input Only & Code Generation p-val \\
\midrule
Logistic Regression & 0.539 (0.055) & 0.540 (0.052) & 0.833 & 0.674 (0.134) & 0.652 (0.128) & 0.033 \\
SVM & 0.514 (0.055) & 0.517 (0.060) & 0.237 & 0.657 (0.150) & 0.638 (0.138) & 0.053 \\
MLP & 0.537 (0.046) & 0.537 (0.041) & 0.992 & 0.652 (0.093) & 0.640 (0.104) & 0.216 \\
Ensemble & 0.535 (0.056) & 0.543 (0.053) & 0.309 & 0.683 (0.127) & 0.673 (0.117) & 0.368 \\
RouteLLM & 0.533 (0.051) & 0.543 (0.044) & 0.091 & 0.654 (0.100) & 0.657 (0.102) & 0.869 \\
\bottomrule
\end{tabular}



#### Embedding

In [14]:
testing_df = load_dataframes_for_prompt_style(is_code_gen=False, type_filter=DEFAULT_EMBEDDING_TYPE)
code_gen_df = load_dataframes_for_prompt_style(is_code_gen=True, type_filter=DEFAULT_EMBEDDING_TYPE)
compute_and_print_statistics_with_wilcoxon(testing_df, code_gen_df)

\begin{tabular}{lrrrrrr}
\toprule
Algorithm & Testing Entire & Testing Input Only & Testing p-val & Code Generation Entire & Code Generation Input Only & Code Generation p-val \\
\midrule
Logistic Regression & 0.522 (0.042) & 0.528 (0.047) & 0.008 & 0.608 (0.107) & 0.608 (0.098) & 0.956 \\
SVM & 0.501 (0.040) & 0.501 (0.047) & 0.863 & 0.609 (0.101) & 0.599 (0.099) & 0.083 \\
MLP & 0.531 (0.050) & 0.528 (0.043) & 0.931 & 0.623 (0.073) & 0.595 (0.071) & $<$ 0.001 \\
Ensemble & 0.527 (0.045) & 0.525 (0.039) & 0.886 & 0.631 (0.100) & 0.625 (0.091) & 0.231 \\
RouteLLM & 0.524 (0.039) & 0.513 (0.037) & 0.045 & 0.625 (0.069) & 0.616 (0.069) & 0.083 \\
\bottomrule
\end{tabular}



In [15]:
# text-embedding-3-small
testing_df = load_dataframes_for_prompt_style(is_code_gen=False, type_filter='text-embedding-3-small')
code_gen_df = load_dataframes_for_prompt_style(is_code_gen=True, type_filter='text-embedding-3-small')
compute_and_print_statistics_with_wilcoxon(testing_df, code_gen_df)

\begin{tabular}{lrrrrrr}
\toprule
Algorithm & Testing Entire & Testing Input Only & Testing p-val & Code Generation Entire & Code Generation Input Only & Code Generation p-val \\
\midrule
Logistic Regression & 0.522 (0.042) & 0.528 (0.047) & 0.017 & 0.608 (0.107) & 0.608 (0.098) & 0.956 \\
SVM & 0.502 (0.040) & 0.501 (0.047) & 0.886 & 0.608 (0.101) & 0.599 (0.098) & 0.114 \\
MLP & 0.529 (0.053) & 0.525 (0.044) & 0.848 & 0.616 (0.069) & 0.590 (0.077) & 0.002 \\
Ensemble & 0.525 (0.046) & 0.523 (0.040) & 0.579 & 0.623 (0.099) & 0.622 (0.094) & 0.648 \\
RouteLLM & 0.524 (0.039) & 0.514 (0.037) & 0.058 & 0.625 (0.069) & 0.615 (0.070) & 0.083 \\
\bottomrule
\end{tabular}



In [16]:
# nomic-embed-text
testing_df = load_dataframes_for_prompt_style(is_code_gen=False, type_filter='nomic-embed-text')
code_gen_df = load_dataframes_for_prompt_style(is_code_gen=True, type_filter='nomic-embed-text')
compute_and_print_statistics_with_wilcoxon(testing_df, code_gen_df)

\begin{tabular}{lrrrrrr}
\toprule
Algorithm & Testing Entire & Testing Input Only & Testing p-val & Code Generation Entire & Code Generation Input Only & Code Generation p-val \\
\midrule
Logistic Regression & 0.484 (0.043) & 0.500 (0.042) & 0.002 & 0.576 (0.097) & 0.569 (0.092) & 0.312 \\
SVM & 0.481 (0.040) & 0.495 (0.054) & 0.003 & 0.596 (0.110) & 0.559 (0.082) & 0.003 \\
MLP & 0.501 (0.038) & 0.510 (0.037) & 0.110 & 0.604 (0.092) & 0.578 (0.074) & 0.006 \\
Ensemble & 0.491 (0.037) & 0.507 (0.038) & 0.001 & 0.598 (0.097) & 0.592 (0.080) & 0.498 \\
RouteLLM & 0.502 (0.029) & 0.508 (0.038) & 0.128 & 0.599 (0.068) & 0.581 (0.069) & 0.004 \\
\bottomrule
\end{tabular}

